# 8주차. 예측 전용 딥러닝 — Time-Series-Library 4종 비교 (GPU)

In [1]:
import sys, json, time, platform, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))
from src.backtest import rolling_backtest
from src.metrics import compute_naive_denominators, score_backtest
from src import bench_models as bm, tslib_adapter as ta

OUT = ROOT / "results/wk8"
PRED_DIR = OUT / "preds"
PRED_DIR.mkdir(parents=True, exist_ok=True)
RUN_LOG, FIT_LOG = OUT / "run_log.csv", OUT / "fit_log.csv"

assert torch.cuda.is_available(), "GPU가 안 보입니다 — 커널이 .venv(Python 3.11)인지 확인"
assert (ta.TSLIB_DIR / "models/TimeXer.py").exists(), f"TSLib 클론 필요: {ta.TSLIB_DIR}"
tslib_commit = subprocess.run(["git", "-C", str(ta.TSLIB_DIR), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip()
if not "4e938a1".startswith(tslib_commit[:7]):
    print(f"⚠ TSLib 커밋 {tslib_commit} — requirements_stage2.txt 핀은 4e938a1")

N_WINDOWS, STEP_SIZE = 52, 168  # 2주차 하네스 그대로 (2017년 52창, expanding)
SLIDING_LEN = 8760              # 8-3b 반대 설계: 1년 sliding
SEED = 2021                     # TSLib run.py fix_seed
LOOKBACKS, H_MAIN = [96, 336, 512], 168

data = bm.load_data()
weather, building_site = bm.load_weather(data)
temp = ta.temperature_frame(data, weather, building_site)

env = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
       "gpu": torch.cuda.get_device_name(0), "tslib_commit": tslib_commit, "train": ta.TRAIN,
       "base_cfg": ta.BASE_CFG, "arch": ta.ARCH, "lr": ta.LR, "seed": SEED}
with open(OUT / "env.json", "w", encoding="utf-8") as f:
    json.dump(env, f, ensure_ascii=False, indent=2)
print(json.dumps(env, ensure_ascii=False, indent=1))
print("데이터 shape:", data.shape)

{
 "python": "3.11.18",
 "torch": "2.10.0+cu130",
 "cuda": "13.0",
 "gpu": "NVIDIA GB10",
 "tslib_commit": "4e938a1",
 "train": {
  "train_epochs": 10,
  "patience": 3,
  "batch_size": 32,
  "val_ratio": 0.1
 },
 "base_cfg": {
  "task_name": "long_term_forecast",
  "label_len": 48,
  "d_model": 512,
  "n_heads": 8,
  "e_layers": 2,
  "d_layers": 1,
  "d_ff": 2048,
  "moving_avg": 25,
  "factor": 1,
  "dropout": 0.1,
  "embed": "timeF",
  "freq": "h",
  "activation": "gelu",
  "use_norm": 1,
  "patch_len": 16,
  "output_attention": false,
  "num_class": 1
 },
 "arch": {
  "DLinear": {},
  "PatchTST": {
   "e_layers": 2,
   "factor": 3
  },
  "iTransformer": {
   "e_layers": 3,
   "d_ff": 512,
   "factor": 3
  },
  "TimeXer": {
   "e_layers": 3,
   "factor": 3
  }
 },
 "lr": {
  "iTransformer": 0.0005
 },
 "seed": 2021
}
데이터 shape: (17544, 17)


/home/blueworks1/kimdongho/Self_Directed_Project/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


## 과제 8-1. BDG2 → Time-Series-Library 어댑터 점검

In [2]:
if str(ta.TSLIB_DIR) not in sys.path:  # ta.build_model 도 같은 경로를 추가함
    sys.path.insert(0, str(ta.TSLIB_DIR))
from utils.timefeatures import time_features  # TSLib

# (a) 시간 특징
assert np.allclose(ta.time_marks(data.index), time_features(data.index, freq="h").T, atol=1e-6)
print("(a) time_marks == TSLib time_features(freq='h') ✓")

# (b) 모델 생성·출력 형태
rows = []
for L in LOOKBACKS:
    for name in ta.MODELS:
        for mode, C in [("M", 17)] + ([("MS", 2)] if name == "TimeXer" else []):
            m = ta.build_model(name, L, H_MAIN, C, mode).cuda().eval()
            with torch.no_grad():
                out = m(torch.randn(2, L, C).cuda(), torch.zeros(2, L, 4).cuda(), None, None)
            rows.append({"model": name, "mode": mode, "seq_len": L, "출력 shape": tuple(out.shape),
                         "파라미터 수": sum(p.numel() for p in m.parameters())})
print("(b)")
print(pd.DataFrame(rows).to_string(index=False))

# (c) 2창 백테스트 (매 창 재학습) → 하네스 형식·점수 확인
mf, pf, log = ta.make_tslib_fns("DLinear", 336, H_MAIN, refit_hours=168, seed=SEED)
res = rolling_backtest(data, mf, pf, n_windows=2, horizon=H_MAIN, train_mode="expanding", step_size=STEP_SIZE)
print("(c)", res.shape, "| NaN 예측:", res["y_pred"].isna().sum())
print(res.head().to_string(index=False))
print(pd.DataFrame(log["fits"]).round(3).to_string(index=False))

(a) time_marks == TSLib time_features(freq='h') ✓
(b)
       model mode  seq_len     출력 shape   파라미터 수
     DLinear    M       96 (2, 168, 17)    32592
    PatchTST    M       96 (2, 168, 17)  7346344
iTransformer    M       96 (2, 168, 17)  4870824
     TimeXer    M       96 (2, 168, 17) 13281960
     TimeXer   MS       96  (2, 168, 1) 13273768
     DLinear    M      336 (2, 168, 17)   113232
    PatchTST    M      336 (2, 168, 17)  9926824
iTransformer    M      336 (2, 168, 17)  4993704
     TimeXer    M      336 (2, 168, 17) 14695080
     TimeXer   MS      336  (2, 168, 1) 14686888
     DLinear    M      512 (2, 168, 17)   172368
    PatchTST    M      512 (2, 168, 17) 11819176
iTransformer    M      512 (2, 168, 17)  5083816
     TimeXer    M      512 (2, 168, 17) 15731368
     TimeXer   MS      512  (2, 168, 1) 15723176
(c) (5712, 5) | NaN 예측: 0
 window               building           timestamp  y_true    y_pred
      0 Hog_parking_Antoinette 2017-12-18 00:00:00    22.5 18.89446

## 실험 계획 + 학습 1회 시간 측정 → 재학습 주기 결정

In [2]:
ONCE = 10**6  # 사실상 재학습 없음 = 첫 평가 시점 이전 데이터로 1회 학습 (교재 기준)

PLAN = (
    # 8-2 · 8-3a: look-back 3종 × 4종, 168h, 1회 학습  — 8-2 표는 이 중 look-back 336
    [dict(task="8-2/8-3a", model=m, seq_len=L, horizon=H_MAIN) for L in LOOKBACKS for m in ta.MODELS]
    # 8-3b: look-back 336, 8주마다 재학습 — expanding / sliding 을 같은 조건으로 (train_mode 만 다름)
    + [dict(task="8-3b", model=m, seq_len=336, horizon=H_MAIN, train_mode=tm, refit_hours=1344)
       for tm in ["expanding", "sliding"] for m in ta.MODELS]
    # 8-4: TimeXer 건물별(G=17) — 전력만(S) vs [외기온, 전력](MS), 1회 학습
    + [dict(task="8-4", model="TimeXer", seq_len=336, horizon=h, mode=md) for h in [24, 168] for md in ["S", "MS"]]
    # 8-5: 누적 비교표용 24h (look-back 336), 1회 학습
    + [dict(task="8-5", model=m, seq_len=336, horizon=24) for m in ta.MODELS]
)
SEED_PLAN = [dict(task="seed", model=m, seq_len=336, horizon=H_MAIN, seed=s) for s in [0, 1] for m in ta.MODELS]
print(f"본 실험 {len(PLAN)}개 + (선택) seed 편차 {len(SEED_PLAN)}개")

SMOKE = OUT / "smoke_timing.csv"
if not SMOKE.exists():
    longest = data.iloc[:-H_MAIN]
    rows = []
    for name, L, mode in [(m, 512, "M") for m in ta.MODELS] + [("TimeXer", 336, "MS")]:
        mf, pf, log = ta.make_tslib_fns(name, L, H_MAIN, mode=mode, temp=temp, seed=SEED)
        pf(mf(longest), H_MAIN)
        rows.append({"model": name, "mode": mode, "seq_len": L, **log["fits"][0], "추론 1회(s)": log["predict_s"]})
        print(rows[-1]["model"], mode, L, f"{rows[-1]['학습 시간(s)']:.1f}s", f"에폭 {rows[-1]['실행 에폭']}",
              f"VRAM {rows[-1]['최대 VRAM(GB)']:.2f}GB")
        torch.cuda.empty_cache()
    pd.DataFrame(rows).to_csv(SMOKE, index=False, encoding="utf-8")
smoke = pd.read_csv(SMOKE)
print(smoke[["model", "mode", "seq_len", "학습 행 수", "실행 에폭", "best 에폭", "학습 시간(s)", "최대 VRAM(GB)", "파라미터 수"]]
      .round(2).to_string(index=False))

# 예상 소요 (상한: 측정값 = look-back 512 · 최장 구간 17,376행 기준)
# 1회 학습은 첫 평가 시점 이전 약 8,800행 → ×0.5, 재학습 expanding 은 평균 ×0.75, sliding 은 ×0.5
fit_s = smoke[smoke["mode"] == "M"].set_index("model")["학습 시간(s)"]
ms_s = smoke.loc[smoke["mode"] == "MS", "학습 시간(s)"].item()
def est_s(p):
    base = ms_s if p.get("mode", "M") != "M" else fit_s[p["model"]]
    r = p.get("refit_hours", ONCE)
    if r == ONCE:
        return base * 0.5
    n_fits = np.ceil(N_WINDOWS / (r // STEP_SIZE))
    return base * n_fits * (0.5 if p.get("train_mode") == "sliding" else 0.75)
est = pd.DataFrame([{"task": p["task"], "예상(분)": est_s(p) / 60} for p in PLAN]).groupby("task", sort=False).sum()
print()
print(est.round(1).to_string())
print(f"합계 약 {est['예상(분)'].sum() / 60:.1f}시간 (상한 추정 — look-back 96·336 은 이보다 빠름)")

본 실험 28개 + (선택) seed 편차 8개
       model mode  seq_len  학습 행 수  실행 에폭  best 에폭  학습 시간(s)  최대 VRAM(GB)   파라미터 수
     DLinear    M      512   17376     10       10      9.75         0.11   172368
    PatchTST    M      512   17376      4        1    464.94         5.86 11819176
iTransformer    M      512   17376      6        3     23.79         0.23  5083816
     TimeXer    M      512   17376      4        1    530.63         5.11 15731368
     TimeXer   MS      336   17376      5        2    577.53         4.29 14686888

          예상(분)
task           
8-2/8-3a   25.7
8-3b      150.1
8-4        19.3
8-5         8.6
합계 약 3.4시간 (상한 추정 — look-back 96·336 은 이보다 빠름)


In [3]:
def run_name(model, seq_len, horizon, mode="M", train_mode="expanding", seed=SEED, refit_hours=ONCE, **_):
    tag = "" if refit_hours == ONCE else f"_r{refit_hours}"
    return f"{model}_L{seq_len}_{horizon}h_{mode}_{train_mode}_s{seed}{tag}"

def mase(res):
    denom = compute_naive_denominators(data, train_end=data.index.get_loc(res["timestamp"].min()), season=168)
    return score_backtest(res, denom)[1]["MASE_평균"]

def run(task, model, seq_len, horizon, mode="M", train_mode="expanding", seed=SEED, refit_hours=ONCE):
    name = run_name(model, seq_len, horizon, mode, train_mode, seed, refit_hours)
    path = PRED_DIR / f"{name}.parquet"
    if path.exists():
        return
    model_fn, predict_fn, log = ta.make_tslib_fns(model, seq_len, horizon, mode=mode, temp=temp,
                                                  refit_hours=refit_hours, seed=seed)
    t0 = time.perf_counter()
    res = rolling_backtest(data, model_fn, predict_fn, n_windows=N_WINDOWS, horizon=horizon,
                           train_mode=train_mode, sliding_train_len=SLIDING_LEN, step_size=STEP_SIZE)
    total = time.perf_counter() - t0
    fits = pd.DataFrame(log["fits"]).assign(run=name)
    row = {"run": name, "task": task, "model": model, "seq_len": seq_len, "horizon": horizon, "mode": mode,
           "train_mode": train_mode, "seed": seed, "refit_hours": refit_hours, "MASE_평균": mase(res),
           "재학습 횟수": len(fits), "학습 1회 평균(s)": fits["학습 시간(s)"].mean(), "학습 시간 합(s)": fits["학습 시간(s)"].sum(),
           "평균 실행 에폭": fits["실행 에폭"].mean(), "최대 VRAM(GB)": fits["최대 VRAM(GB)"].max(),
           "파라미터 수": fits["파라미터 수"].iloc[0], "추론 시간 합(s)": log["predict_s"], "백테스트 전체(s)": total}
    fits.to_csv(FIT_LOG, mode="a", header=not FIT_LOG.exists(), index=False, encoding="utf-8")
    pd.DataFrame([row]).to_csv(RUN_LOG, mode="a", header=not RUN_LOG.exists(), index=False, encoding="utf-8")
    res.to_parquet(path, index=False)  # 마지막에 저장 → parquet 가 있으면 로그도 있음
    print(f"{name}: MASE {row['MASE_평균']:.4f} | 학습 {len(fits)}회 · 평균 {row['학습 1회 평균(s)']:.1f}s · "
          f"VRAM {row['최대 VRAM(GB)']:.2f}GB | 전체 {total / 60:.1f}분")
    torch.cuda.empty_cache()

def run_task(task, plan=PLAN):
    for p in plan:
        if p["task"] == task:
            run(**p)

## 과제 8-2 · 8-3a. 4종 동일 조건 × look-back 96 / 336 / 512 (168h, expanding)

In [5]:
run_task("8-2/8-3a")

PatchTST_L336_168h_M_expanding_s2021: MASE 0.8901 | 학습 1회 · 평균 336.7s · VRAM 4.56GB | 전체 5.6분
TimeXer_L336_168h_M_expanding_s2021: MASE 0.9510 | 학습 1회 · 평균 225.6s · VRAM 4.18GB | 전체 3.8분
PatchTST_L512_168h_M_expanding_s2021: MASE 0.8958 | 학습 1회 · 평균 506.9s · VRAM 5.79GB | 전체 8.5분
TimeXer_L512_168h_M_expanding_s2021: MASE 0.9173 | 학습 1회 · 평균 257.0s · VRAM 5.03GB | 전체 4.3분


In [4]:
run_task("8-3b")

TimeXer_L336_168h_M_sliding_s2021_r1344: MASE 0.9358 | 학습 7회 · 평균 269.3s · VRAM 4.32GB | 전체 31.4분


In [5]:
log = pd.read_csv(RUN_LOG).drop_duplicates("run", keep="last")
print(log[log["task"] == "8-3b"][["model", "train_mode", "MASE_평균", "재학습 횟수", "학습 1회 평균(s)", "최대 VRAM(GB)"]]
      .round(4).to_string(index=False))

       model train_mode  MASE_평균  재학습 횟수  학습 1회 평균(s)  최대 VRAM(GB)
     DLinear  expanding   1.0250       7       7.1057       0.0319
    PatchTST  expanding   0.8885       7     589.0676       4.6552
iTransformer  expanding   0.9152       7      10.2137       0.2004
     TimeXer  expanding   0.9127       7     356.9317       4.3144
     DLinear    sliding   1.0480       7       2.9561       0.0294
    PatchTST    sliding   0.9207       7     325.2059       4.6548
iTransformer    sliding   0.9761       7       5.6882       0.1985
     TimeXer    sliding   0.9358       7     269.3269       4.3172


## 과제 8-4. TimeXer 외기온 공변량 (건물별 모델, look-back 336)

In [6]:
run_task("8-4")

TimeXer_L336_24h_S_expanding_s2021: MASE 0.5963 | 학습 1회 · 평균 549.2s · VRAM 4.18GB | 전체 9.2분
TimeXer_L336_24h_MS_expanding_s2021: MASE 0.6205 | 학습 1회 · 평균 428.9s · VRAM 4.19GB | 전체 7.2분
TimeXer_L336_168h_S_expanding_s2021: MASE 0.8816 | 학습 1회 · 평균 471.7s · VRAM 4.21GB | 전체 7.9분
TimeXer_L336_168h_MS_expanding_s2021: MASE 0.8811 | 학습 1회 · 평균 532.4s · VRAM 4.22GB | 전체 8.9분


## 과제 8-5. 24h 4종 (look-back 336) — 누적 비교표에 24h 열을 채움

In [4]:
run_task("8-5")

PatchTST_L336_24h_M_expanding_s2021: MASE 0.6469 | 학습 1회 · 평균 811.8s · VRAM 4.50GB | 전체 13.6분
TimeXer_L336_24h_M_expanding_s2021: MASE 0.6124 | 학습 1회 · 평균 656.9s · VRAM 4.14GB | 전체 11.0분


## seed 편차 — 순위 역전이 seed 노이즈보다 큰가

In [6]:
run_task("seed", SEED_PLAN)

DLinear_L336_168h_M_expanding_s0: MASE 1.0399 | 학습 1회 · 평균 1.8s · VRAM 0.03GB | 전체 0.0분
PatchTST_L336_168h_M_expanding_s0: MASE 0.9004 | 학습 1회 · 평균 345.2s · VRAM 4.56GB | 전체 5.8분
iTransformer_L336_168h_M_expanding_s0: MASE 0.9323 | 학습 1회 · 평균 5.5s · VRAM 0.16GB | 전체 0.1분
TimeXer_L336_168h_M_expanding_s0: MASE 0.9140 | 학습 1회 · 평균 230.1s · VRAM 4.18GB | 전체 3.8분
DLinear_L336_168h_M_expanding_s1: MASE 1.0427 | 학습 1회 · 평균 2.8s · VRAM 0.03GB | 전체 0.0분
PatchTST_L336_168h_M_expanding_s1: MASE 0.8962 | 학습 1회 · 평균 344.3s · VRAM 4.56GB | 전체 5.7분
iTransformer_L336_168h_M_expanding_s1: MASE 0.9487 | 학습 1회 · 평균 5.3s · VRAM 0.16GB | 전체 0.1분
TimeXer_L336_168h_M_expanding_s1: MASE 0.9632 | 학습 1회 · 평균 233.2s · VRAM 4.18GB | 전체 3.9분
